### Compatibility fix
Before scikit-learn preprocessing, nullable pandas `pd.NA` values in categorical columns are converted to `np.nan`. This prevents the `boolean value of NA is ambiguous` error that can occur when `SimpleImputer` receives pandas nullable string values.

# Bike-Sharing Rental Demand Prediction
## End-to-end project: EDA, Feature Engineering, Model Building, Evaluation & Deployment

**Business objective:** Predict the number of bikes rented (`cnt`) using calendar, time and weather information. Better demand estimates can help a bike-sharing business plan bike availability, operations, pricing and marketing.

**Project scope follows the supplied Business Objective document:**
1. Dataset understanding and data cleaning
2. EDA and visualisation
3. Feature engineering
4. Regression model building
5. Hyperparameter tuning with cross-validation
6. Model evaluation and model comparison
7. Model deployment and monitoring outline

**Dataset facts observed during preparation:** 17,379 rows and 17 columns. The target is `cnt`. The source includes `?` markers and object-typed numeric columns, so the cleaning steps below explicitly handle them.

> Run the notebook from top to bottom. Review the printed outputs and charts, and use the actual evaluation results in your final presentation. Do not claim a model score before running the notebook.

## 1. Import libraries and configure the dataset path

In [ ]:
# If required, install packages in a Jupyter cell:
# %pip install pandas numpy matplotlib seaborn scikit-learn joblib

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance
import joblib

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Your Windows dataset path:
DATA_PATH = r"C:\Users\nikhi\Downloads\Dataset (2) (1).csv"

# Fallback: place the CSV in the same folder as this notebook and name it as below.
FALLBACK_PATH = "Dataset (2) (1).csv"

if os.path.exists(DATA_PATH):
    csv_path = DATA_PATH
elif os.path.exists(FALLBACK_PATH):
    csv_path = FALLBACK_PATH
else:
    raise FileNotFoundError(
        "Dataset not found. Update DATA_PATH with the correct CSV location "
        "or place the CSV beside this notebook."
    )

print("Dataset path:", csv_path)

## 2. Load the dataset

In [ ]:
df_raw = pd.read_csv(csv_path)
df = df_raw.copy()

print("Shape:", df.shape)
display(df.head())
display(df.tail())

## 3. Understand the dataset structure

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.rename("data_type").to_frame())

print("\nDataset information:")
df.info()

print("\nDescriptive statistics (all columns):")
display(df.describe(include="all").T)

## 4. Data quality assessment

In [ ]:
# Count explicit missing values and common text markers such as '?'
print("Null values before cleaning:")
display(df.isna().sum().sort_values(ascending=False).to_frame("null_count"))

print("Question-mark markers before cleaning:")
question_mark_counts = df.astype(str).eq("?").sum()
display(question_mark_counts[question_mark_counts > 0].to_frame("question_mark_count"))

print("Duplicate rows:", df.duplicated().sum())

# Unique values in categorical/object columns help identify inconsistent labels.
for col in df.select_dtypes(include="object").columns:
    print(f"\n{col}: {df[col].nunique(dropna=False)} unique values")
    print(df[col].value_counts(dropna=False).head(20))

## 5. Clean and correct data types

In [ ]:
# Replace '?' strings (and whitespace-only strings) with NaN.
df = df.replace(r"^\s*\?\s*$", np.nan, regex=True)
df = df.replace(r"^\s*$", np.nan, regex=True)

# Parse the date. The supplied data uses day-month-year format.
df["dteday"] = pd.to_datetime(df["dteday"], format="%d-%m-%Y", errors="coerce")

# Convert numeric-looking columns to numeric values.
numeric_candidates = [
    "instant", "yr", "mnth", "hr", "weekday",
    "temp", "atemp", "hum", "windspeed", "casual", "registered", "cnt"
]
for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove exact duplicate rows, if any.
before = len(df)
df = df.drop_duplicates().copy()
print("Duplicate rows removed:", before - len(df))

# Show remaining missing values after parsing/conversion.
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_after_conversion"))
display(df.dtypes.to_frame("data_type"))

### Cleaning notes and assumptions
- The literal `?` values are treated as missing values, not as valid categories.
- Numeric-looking columns are converted to numeric types using coercion; invalid values become missing.
- `dteday` is parsed using the observed `DD-MM-YYYY` format.
- The `season` value `springer` is retained as it appears in the supplied data. Confirm with the data provider whether it is a typo for `spring` before changing it.
- Missing predictor values are imputed inside the model pipeline to avoid fitting imputers on the test set.

## 6. Missing-value analysis and handling

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)
display(missing_summary)

# Target values cannot be imputed reliably for supervised learning.
target_col = "cnt"
df = df.dropna(subset=[target_col, "dteday"]).copy()

print("Shape after dropping rows without target/date:", df.shape)

## 7. Feature engineering

In [ ]:
# Derive calendar features from the date column.
df["year_from_date"] = df["dteday"].dt.year
df["month_from_date"] = df["dteday"].dt.month
df["day_of_month"] = df["dteday"].dt.day
df["day_of_week_from_date"] = df["dteday"].dt.dayofweek
df["is_weekend"] = (df["day_of_week_from_date"] >= 5).astype(int)

# Use the date-derived year/month where available; use original values as fallback.
if "yr" in df.columns:
    df["year_feature"] = df["year_from_date"].fillna(df["yr"])
else:
    df["year_feature"] = df["year_from_date"]

if "mnth" in df.columns:
    df["month_feature"] = df["month_from_date"].fillna(df["mnth"])
else:
    df["month_feature"] = df["month_from_date"]

# Convert binary text flags to consistent categorical strings.
for col in ["holiday", "workingday", "season", "weathersit"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

# IMPORTANT: do not use instant (row identifier), casual or registered as predictors.
# casual + registered = cnt, so using either would leak information about the target.
feature_columns = [
    "season", "holiday", "workingday", "weathersit",
    "hr", "weekday", "temp", "atemp", "hum", "windspeed",
    "year_feature", "month_feature", "day_of_month",
    "day_of_week_from_date", "is_weekend"
]
feature_columns = [c for c in feature_columns if c in df.columns]

X = df[feature_columns].copy()
y = df[target_col].astype(float).copy()

print("Features used:", feature_columns)
print("Target:", target_col)
print("X shape:", X.shape, "| y shape:", y.shape)
display(X.head())

## 8. Exploratory Data Analysis (EDA)

### 8.1 Target variable distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(y, bins=40, kde=True, ax=axes[0])
axes[0].set_title("Distribution of total bike rentals (cnt)")
axes[0].set_xlabel("Total rentals")

sns.boxplot(x=y, ax=axes[1])
axes[1].set_title("Boxplot of total bike rentals")
plt.tight_layout()
plt.show()

print(y.describe())

### 8.2 Rental demand over time

In [ ]:
time_df = df[["dteday", "cnt"]].dropna().sort_values("dteday")
daily_demand = time_df.groupby("dteday", as_index=False)["cnt"].sum()

plt.figure(figsize=(16, 5))
plt.plot(daily_demand["dteday"], daily_demand["cnt"], linewidth=0.8)
plt.title("Daily total bike rental demand over time")
plt.xlabel("Date")
plt.ylabel("Total rentals per day")
plt.tight_layout()
plt.show()

monthly_demand = df.groupby("month_feature")["cnt"].mean().sort_index()
plt.figure(figsize=(10, 4))
sns.lineplot(x=monthly_demand.index, y=monthly_demand.values, marker="o")
plt.title("Average hourly rentals by month")
plt.xlabel("Month")
plt.ylabel("Average rentals per hour")
plt.xticks(range(1, 13))
plt.tight_layout()
plt.show()

### 8.3 Demand by hour, weekday, season and weather

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

sns.lineplot(data=df, x="hr", y="cnt", estimator="mean", errorbar=None, ax=axes[0, 0])
axes[0, 0].set_title("Average rentals by hour")

sns.barplot(data=df, x="weekday", y="cnt", errorbar=None, ax=axes[0, 1])
axes[0, 1].set_title("Average rentals by weekday")

sns.barplot(data=df, x="season", y="cnt", errorbar=None, ax=axes[1, 0])
axes[1, 0].set_title("Average rentals by season")
axes[1, 0].tick_params(axis="x", rotation=20)

sns.barplot(data=df, x="weathersit", y="cnt", errorbar=None, ax=axes[1, 1])
axes[1, 1].set_title("Average rentals by weather")
axes[1, 1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

### 8.4 Scatter plots: weather versus demand

In [ ]:
weather_features = [c for c in ["temp", "atemp", "hum", "windspeed"] if c in df.columns]
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.flat, weather_features):
    sns.scatterplot(data=df.sample(min(4000, len(df)), random_state=RANDOM_STATE),
                    x=col, y="cnt", alpha=0.35, s=18, ax=ax)
    ax.set_title(f"{col} vs bike rentals")
plt.tight_layout()
plt.show()

### 8.5 Correlation heatmap

In [ ]:
numeric_for_corr = df.select_dtypes(include=np.number)
corr = numeric_for_corr.corr(numeric_only=True)

plt.figure(figsize=(13, 9))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f", linewidths=0.3)
plt.title("Correlation matrix of numeric variables")
plt.tight_layout()
plt.show()

if "cnt" in corr.columns:
    print("Correlation with target cnt:")
    display(corr["cnt"].sort_values(ascending=False).to_frame("correlation_with_cnt"))

## 9. Outlier detection

In [ ]:
# IQR-based outlier counts for numerical variables.
# This is diagnostic: tree-based models are generally robust to scaling and can handle
# skewed data. Do not automatically delete high-demand observations; they may be real peaks.
numeric_eda_cols = [c for c in ["hr", "temp", "atemp", "hum", "windspeed", "cnt"] if c in df.columns]
outlier_rows = []
for col in numeric_eda_cols:
    s = pd.to_numeric(df[col], errors="coerce").dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = ((s < lower) | (s > upper)).sum()
    outlier_rows.append([col, lower, upper, int(count), round(count / len(s) * 100, 2)])
outlier_summary = pd.DataFrame(
    outlier_rows, columns=["feature", "lower_bound", "upper_bound", "outlier_count", "outlier_percent"]
)
display(outlier_summary)

## 10. Prepare chronological train-test split

In [ ]:
# Sort chronologically before splitting. The latest 20% of observations are held out
# to represent future demand prediction more realistically than a random split.
model_df = df.dropna(subset=["dteday", target_col]).sort_values(["dteday", "hr"]).copy()
X = model_df[feature_columns].copy()
y = model_df[target_col].astype(float).copy()

# FIX: convert pandas nullable pd.NA values in categorical columns to np.nan.
# This prevents: TypeError: boolean value of NA is ambiguous
for col in X.select_dtypes(include=["object", "string", "category"]).columns:
    X[col] = X[col].astype(object)
    X[col] = X[col].where(pd.notna(X[col]), np.nan)

split_index = int(len(model_df) * 0.80)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print("Training period:", model_df["dteday"].iloc[0], "to", model_df["dteday"].iloc[split_index-1])
print("Testing period :", model_df["dteday"].iloc[split_index], "to", model_df["dteday"].iloc[-1])
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

## 11. Preprocessing pipeline

In [ ]:
# Defensive conversion before sklearn imputers (covers pandas nullable dtypes).
for frame in (X_train, X_test):
    for col in frame.select_dtypes(include=["object", "string", "category"]).columns:
        frame[col] = frame[col].astype(object)
        frame[col] = frame[col].where(pd.notna(frame[col]), np.nan)

categorical_features = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
numerical_features = [c for c in X_train.columns if c not in categorical_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_transformer, numerical_features),
    ("categorical", categorical_transformer, categorical_features)
])

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

## 12. Build and compare regression models

In [ ]:
models = {
    "Decision Tree": DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=15),
    "Random Forest": RandomForestRegressor(
        n_estimators=250, random_state=RANDOM_STATE, n_jobs=-1, max_features=0.9
    ),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE)
}

def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_true, y_pred)
    }

baseline_results = []
fitted_models = {}
predictions = {}

for name, estimator in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted_models[name] = pipe
    predictions[name] = pred
    baseline_results.append({"Model": name, **regression_metrics(y_test, pred)})

baseline_results_df = pd.DataFrame(baseline_results).sort_values("RMSE")
display(baseline_results_df)

## 13. Model comparison visualisations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=baseline_results_df, x="Model", y="RMSE", ax=axes[0])
axes[0].set_title("Model comparison - RMSE (lower is better)")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(data=baseline_results_df, x="Model", y="R2", ax=axes[1])
axes[1].set_title("Model comparison - R-squared (higher is better)")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

## 14. Hyperparameter tuning with time-series cross-validation

In [ ]:
# TimeSeriesSplit preserves order in each fold. Tuning can take several minutes.
tscv = TimeSeriesSplit(n_splits=3)

tuning_spaces = {
    "Decision Tree": {
        "model__max_depth": [5, 8, 12, 16, 20, None],
        "model__min_samples_split": [2, 5, 10, 20],
        "model__min_samples_leaf": [1, 2, 4, 8]
    },
    "Random Forest": {
        "model__n_estimators": [150, 250, 400],
        "model__max_depth": [10, 15, 20, None],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": [0.7, 0.9, 1.0]
    },
    "Gradient Boosting": {
        "model__n_estimators": [100, 150, 250],
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__max_depth": [2, 3, 4],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
        "model__subsample": [0.8, 1.0]
    }
}

tuned_models = {}
tuning_results = []

for name, params in tuning_spaces.items():
    estimator = models[name]
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=params,
        n_iter=12,  # increase to 25-50 for a more exhaustive search if time permits
        scoring="neg_root_mean_squared_error",
        cv=tscv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1
    )
    search.fit(X_train, y_train)
    tuned_models[name] = search.best_estimator_
    tuning_results.append({
        "Model": name,
        "Best CV RMSE": -search.best_score_,
        "Best Parameters": search.best_params_
    })

tuning_results_df = pd.DataFrame(tuning_results).sort_values("Best CV RMSE")
display(tuning_results_df)

## 15. Evaluate tuned models on the held-out test period

In [ ]:
tuned_test_results = []
tuned_predictions = {}

for name, pipe in tuned_models.items():
    pred = pipe.predict(X_test)
    tuned_predictions[name] = pred
    tuned_test_results.append({"Model": name, **regression_metrics(y_test, pred)})

tuned_results_df = pd.DataFrame(tuned_test_results).sort_values("RMSE")
display(tuned_results_df)

## 16. Select the best model

In [ ]:
# Select by lowest test RMSE. Review MAE and R2 as supporting metrics.
best_model_name = tuned_results_df.iloc[0]["Model"]
best_model = tuned_models[best_model_name]
best_prediction = tuned_predictions[best_model_name]

print("Selected model:", best_model_name)
print("Reason: lowest RMSE on the chronological hold-out test set.")
print("Test metrics:")
display(tuned_results_df[tuned_results_df["Model"] == best_model_name])

## 17. Actual vs predicted and residual analysis

In [ ]:
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": best_prediction
}, index=y_test.index)

plt.figure(figsize=(7, 7))
sns.scatterplot(data=comparison, x="Actual", y="Predicted", alpha=0.35)
lims = [
    min(comparison["Actual"].min(), comparison["Predicted"].min()),
    max(comparison["Actual"].max(), comparison["Predicted"].max())
]
plt.plot(lims, lims, linestyle="--")
plt.title(f"Actual vs Predicted - {best_model_name}")
plt.tight_layout()
plt.show()

residuals = comparison["Actual"] - comparison["Predicted"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(residuals, bins=40, kde=True, ax=axes[0])
axes[0].set_title("Residual distribution")
sns.scatterplot(x=comparison["Predicted"], y=residuals, alpha=0.35, ax=axes[1])
axes[1].axhline(0, linestyle="--")
axes[1].set_title("Residuals vs predicted")
axes[1].set_xlabel("Predicted rentals")
axes[1].set_ylabel("Residual")
plt.tight_layout()
plt.show()

display(comparison.head(15))

## 18. Feature importance (permutation importance)

In [ ]:
# Permutation importance works with the full pipeline and original feature columns.
perm = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=5, random_state=RANDOM_STATE,
    scoring="neg_root_mean_squared_error", n_jobs=-1
)
importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": perm.importances_mean,
    "Std": perm.importances_std
}).sort_values("Importance", ascending=False)

display(importance_df)
plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(15), x="Importance", y="Feature")
plt.title("Top feature importance (permutation)")
plt.tight_layout()
plt.show()

## 19. Save the trained model for deployment

In [ ]:
import os
import json
from datetime import datetime

os.makedirs("deployment_artifacts", exist_ok=True)

MODEL_FILE = os.path.join("deployment_artifacts", "bike_demand_model.joblib")
METRICS_FILE = os.path.join("deployment_artifacts", "model_metrics.json")
FEATURES_FILE = os.path.join("deployment_artifacts", "feature_columns.json")

joblib.dump(best_model, MODEL_FILE)

best_metrics = regression_metrics(y_test, best_prediction)
with open(METRICS_FILE, "w", encoding="utf-8") as f:
    json.dump({
        "selected_model": best_model_name,
        "test_metrics": best_metrics,
        "saved_at": datetime.now().isoformat()
    }, f, indent=4)

with open(FEATURES_FILE, "w", encoding="utf-8") as f:
    json.dump(feature_columns, f, indent=4)

print("Saved model:", MODEL_FILE)
print("Saved metrics:", METRICS_FILE)
print("Saved feature list:", FEATURES_FILE)

## 20. Test deployment prediction with a sample input

In [ ]:
# Build one sample from an actual test row to ensure every required input column exists.
sample_input = X_test.iloc[[0]].copy()
sample_prediction = best_model.predict(sample_input)[0]

print("Sample input:")
display(sample_input)
print(f"Predicted hourly bike rentals: {max(0, sample_prediction):.0f}")
print(f"Actual hourly bike rentals: {y_test.iloc[0]:.0f}")

## 21. Create a Streamlit deployment application (app.py)

The following cell writes a simple user interface. It loads the saved pipeline and accepts calendar/weather inputs. Run it from the same project folder after installing Streamlit:

```bash
pip install streamlit joblib pandas scikit-learn
streamlit run app.py
```

In [ ]:
app_code = r"""
import streamlit as st
import pandas as pd
import joblib
import json
import os

st.set_page_config(page_title="Bike Demand Prediction", page_icon="🚲", layout="centered")
st.title("🚲 Bike-Sharing Rental Demand Prediction")
st.write("Estimate hourly bike rental demand using time, calendar and weather conditions.")

MODEL_PATH = os.path.join("deployment_artifacts", "bike_demand_model.joblib")
FEATURES_PATH = os.path.join("deployment_artifacts", "feature_columns.json")

if not os.path.exists(MODEL_PATH):
    st.error("Model file not found. Run the Jupyter notebook through the model-saving section first.")
    st.stop()

model = joblib.load(MODEL_PATH)
with open(FEATURES_PATH, "r", encoding="utf-8") as f:
    feature_columns = json.load(f)

with st.form("prediction_form"):
    date = st.date_input("Date")
    hour = st.slider("Hour of day", 0, 23, 8)
    weekday = st.selectbox("Weekday (0=Monday ... 6=Sunday)", list(range(7)), index=0)
    season = st.selectbox("Season", ["springer", "summer", "fall", "winter"])
    holiday = st.selectbox("Holiday", ["No", "Yes"])
    workingday = st.selectbox("Working day", ["Working Day", "No work"])
    weather = st.selectbox("Weather", ["Clear", "Mist", "Light Snow", "Heavy Rain"])
    temp = st.slider("Temperature (normalised)", 0.0, 1.0, 0.5)
    atemp = st.slider("Feels-like temperature (normalised)", 0.0, 1.0, 0.5)
    hum = st.slider("Humidity (normalised)", 0.0, 1.0, 0.5)
    windspeed = st.slider("Windspeed (normalised)", 0.0, 1.0, 0.2)
    submitted = st.form_submit_button("Predict demand")

if submitted:
    dt = pd.Timestamp(date)
    values = {
        "season": season,
        "holiday": holiday,
        "workingday": workingday,
        "weathersit": weather,
        "hr": hour,
        "weekday": weekday,
        "temp": temp,
        "atemp": atemp,
        "hum": hum,
        "windspeed": windspeed,
        "year_feature": dt.year,
        "month_feature": dt.month,
        "day_of_month": dt.day,
        "day_of_week_from_date": dt.dayofweek,
        "is_weekend": int(dt.dayofweek >= 5)
    }
    input_df = pd.DataFrame([{col: values.get(col) for col in feature_columns}])
    prediction = max(0, float(model.predict(input_df)[0]))
    st.success(f"Estimated bike rentals for this hour: {prediction:.0f}")
    st.caption("This is a model estimate, not a guaranteed count. Monitor real-world errors.")
"""
with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("Created app.py in:", os.path.abspath("app.py"))

## 22. Deployment monitoring and maintenance

After deployment, the business should:
- Store actual rental counts and model predictions with timestamps.
- Calculate MAE and RMSE regularly when actual outcomes become available.
- Compare prediction errors by hour, season, weather and working-day status.
- Watch for changes in demand patterns, weather distributions and input quality (data drift).
- Retrain using recent, representative data when performance degrades or operating conditions change.
- Version the model, preprocessing pipeline, training data period and evaluation metrics.
- Validate input ranges and reject or flag invalid user inputs.

**Deployment limitation:** This notebook creates a local Streamlit prototype and saves a reusable model. It does not publish the app to a public cloud/server. Public deployment requires a hosting platform, dependency file, access controls and operational monitoring.

## 23. Final project conclusion template

After running all cells, complete this section using your actual output values:

- **Business problem:** Estimate hourly bike-sharing demand to support availability and operational planning.
- **Dataset:** Report the final cleaned row/column counts and date coverage printed above.
- **EDA insights:** Summarise the observed hourly, weekday, seasonal and weather-related demand patterns from your charts.
- **Models compared:** Decision Tree Regressor, Random Forest Regressor and Gradient Boosting Regressor.
- **Validation:** Chronological 80/20 hold-out split with time-series cross-validation for tuning.
- **Best model:** Insert the selected model name printed in Section 16.
- **Performance:** Insert the test MAE, MSE, RMSE and R² from Section 16.
- **Deployment:** Saved the fitted preprocessing + regression pipeline as a Joblib file and created a Streamlit prediction interface.
- **Future improvement:** Test additional models, collect current operational data, tune more extensively and deploy on a managed hosting platform.

## Appendix: Key project participation practices from the supplied guidelines

The participation guidelines require active contribution, timely task completion, clear communication, daily team connectivity, sharing insights, collaboration, active listening and recognition of teammates' contributions. Teams are expected to present progress in weekly reviews. Technical issues should first be discussed within the group, then with other groups, and finally escalated to the mentor. Attendance is mandatory, and the guidelines state that missing more than two meetings can result in removal from the project.

These are team-process requirements rather than modelling steps, so they are documented here instead of being mixed into the machine-learning pipeline.